In [ ]:
"""3D CNN model for Alzheimer's disease detection from structural MRI.

Reference:
    Liu et al. "On the design of convolutional neural networks for automatic
    detection of Alzheimer's disease." ML4H @ NeurIPS 2019.
    https://arxiv.org/abs/1911.03740

    Liu et al. "Generalizable deep learning model for early Alzheimer's
    disease detection from structural MRIs." Scientific Reports, 2022.
    https://www.nature.com/articles/s41598-022-20674-x
"""

from typing import Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from pyhealth.datasets import SampleDataset
from pyhealth.models import BaseModel


class AlzheimerCNN(BaseModel):
    def __init__(
        self,
        dataset: SampleDataset,
        feature_keys: List[str],
        label_key: str,
        mode: str,
        init_channels: int = 32,
        classifier_hidden_dim: int = 128,
        dropout: float = 0.5,
        **kwargs,
    ) -> None:
        """
        PyHealth BaseModels require:
        dataset, feature_keys, and label_key -> dataloader
        """
        super(AlzheimerCNN, self).__init__(
            dataset=dataset,
            feature_keys=feature_keys,
            label_key=label_key,
            mode=mode,
        )

        self.init_channels = init_channels
        self.classifier_hidden_dim = classifier_hidden_dim

        # ── Block 1: Conv2d(1 → C) → InstanceNorm → LeakyReLU → MaxPool ──
        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 2: Conv2d(C → 2C) → InstanceNorm → LeakyReLU → MaxPool ─
        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 3: Conv2d(2C → 4C) → InstanceNorm → LeakyReLU → MaxPool ─
        self.block3 = nn.Sequential(
            nn.Conv2d(init_channels * 2, init_channels * 4, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 4),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 4: Conv2d(4C → 8C) → InstanceNorm → LeakyReLU → AdaptiveAvgPool ─
        self.block4 = nn.Sequential(
            nn.Conv2d(init_channels * 4, init_channels * 8, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 8),
            nn.LeakyReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # ── Classifier: FC(8C → hidden_dim) → LeakyReLU → Dropout → FC(hidden_dim → output) ─
        output_size = self.get_output_size(self.label_tokenizer)
        self.classifier = nn.Sequential(
            nn.Linear(init_channels * 8, classifier_hidden_dim),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(classifier_hidden_dim, output_size),
        )